# Energy Demand Analytics: Synthetic Meter Data and Load Prediction

**Type:** Original supplementary practice project  
**Domain:** Electrical engineering and energy analytics  
**Data:** Simulated 30-minute meter readings

This example connects data science methods with energy systems. It is a demonstration using synthetic data—not BUET thesis data, field measurements, a utility dataset or a claim about actual Bangladesh demand.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rng = np.random.default_rng(91)
dates = pd.date_range("2025-01-01", periods=48*60, freq="30min")
hour = dates.hour + dates.minute/60
weekday = dates.dayofweek
is_weekend = (weekday >= 5).astype(int)
temperature = 27 + 5*np.sin(2*np.pi*(dates.dayofyear.to_numpy()-90)/365.25) + rng.normal(0,1.2,len(dates))
morning_peak = 18*np.exp(-0.5*((hour-9)/2.0)**2)
evening_peak = 30*np.exp(-0.5*((hour-19)/2.5)**2)
daily_base = 42 + 8*(1-is_weekend)
cooling_load = np.maximum(temperature-24,0)*2.2
load_kw = daily_base + morning_peak + evening_peak + cooling_load + rng.normal(0,4,len(dates))
meter = pd.DataFrame({
    "timestamp":dates,
    "load_kw":np.maximum(load_kw,5),
    "temperature_c":temperature,
    "hour":hour,
    "day_of_week":weekday,
    "is_weekend":is_weekend
})
meter.head()

## 1. Understand the load profile

In [ ]:
daily = meter.set_index("timestamp")["load_kw"].resample("D").mean()
hourly_profile = meter.groupby("hour")["load_kw"].mean()

fig, ax = plt.subplots(1,2,figsize=(13,4))
ax[0].plot(daily.index, daily.values)
ax[0].set(title="Daily average load", xlabel="Date", ylabel="kW")
ax[1].plot(hourly_profile.index, hourly_profile.values, marker="o")
ax[1].set(title="Average load by half-hour clock time", xlabel="Hour", ylabel="kW")
for a in ax: a.grid(alpha=.2)
plt.tight_layout()
plt.show()

display(meter[["load_kw","temperature_c"]].describe().round(2))

A random split can leak future information into the training set for time-dependent data. Here the last 20% of observations are held out chronologically. Calendar variables are known in advance; lagged load features are deliberately omitted in this introductory example. **The holdout temperature values are also supplied as inputs in this demonstration**—a real forward forecast would need a temperature forecast or a scenario assumption rather than future observed temperature.


In [ ]:
features = meter[["temperature_c","hour","day_of_week","is_weekend"]].copy()
target = meter["load_kw"].copy()
split_at = int(len(meter)*.8)
X_train, X_test = features.iloc[:split_at], features.iloc[split_at:]
y_train, y_test = target.iloc[:split_at], target.iloc[split_at:]

model = RandomForestRegressor(
    n_estimators=250, min_samples_leaf=3, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)
prediction = model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test,prediction),3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test,prediction)),3))
print("R2:", round(r2_score(y_test,prediction),3))

## 3. Compare actual and predicted values

In [ ]:
sample = pd.DataFrame({
    "actual":y_test.iloc[:7*48].to_numpy(),
    "predicted":prediction[:7*48]
}, index=meter["timestamp"].iloc[split_at:split_at+7*48])

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(sample.index, sample["actual"], label="Actual", linewidth=1)
ax.plot(sample.index, sample["predicted"], label="Predicted", linewidth=1)
ax.set(title="One-week holdout: synthetic meter load", xlabel="Timestamp", ylabel="kW")
ax.legend()
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

## 4. Feature importance and engineering interpretation

In [ ]:
importance = pd.Series(model.feature_importances_, index=X_train.columns).sort_values()
importance.plot(kind="barh", figsize=(7,4), title="Random Forest feature importance")
plt.xlabel("Impurity-based importance")
plt.tight_layout()
plt.show()
display(importance.sort_values(ascending=False).to_frame("importance"))

## Next steps and limitations

For a more defensible energy forecasting study, include lagged demand and rolling statistics carefully, test for seasonality and structural breaks, benchmark against persistence and seasonal-naive forecasts, use rolling-origin evaluation, quantify prediction intervals, and document meter sampling and missing-data handling.

Do not infer real consumer behaviour, grid peaks or energy policy conclusions from this synthetic dataset. Real deployments require validated meter data, data governance and engineering review.